# Credit Portfolio Optimization
This project is used to optimize the credit portfolio by minimizing the default risk of the entire portfolio.

In [168]:
import numpy as np
import pandas as pd

###  Data Description
The methodology discussed above is applied to optimize CDS portfolios. The portfolio consists of a single-name CDS from the Markit MARKIT CDX.NA.IG.37. The single names and market tickers as follows, alphabetically: 

We select the first eight of them. The daily closing prices of five-year CDS are used for liquidity reasons. The sample period is from January 2, 2014, to December 31, 2021, with a total of 2002 observations for each single name CDS.

In [169]:
df = pd.read_csv('./stock_data.csv')
df = df.iloc[:]  
df.set_index('Date',inplace=True)
df.dropna(axis=0,inplace=True)
df.shape       # 2002 rows * 121 columns


(1258, 100)

In [170]:
df.head()

,ATHM,V,TSLA,VIS,AAPL,ABEV,CF,AIG,AMAT,AMD,...,TGT,TSM,TWTR,TXN,VZ,WFC,WMB,XOM,YELP,VLO
Date,,,,,,,,,,,,,,,,,,,,,
2014/2/18,35.825970,53.199341,203.699997,89.915695,68.972733,5.715869,34.519791,45.755302,17.419250,3.70,...,48.102768,15.182836,58.180000,38.632580,36.576538,39.911037,30.804720,78.703629,91.739998,42.936657
2014/2/19,35.994167,52.686302,193.639999,89.138504,67.883812,5.732880,36.264172,44.916264,17.326839,3.72,...,48.887573,15.157234,55.500000,38.562363,37.014050,39.391911,30.774330,78.603218,90.000000,42.733562
2014/2/20,34.886044,52.587467,209.970001,90.107704,67.098045,5.928513,36.758640,44.888901,17.548626,3.69,...,48.298965,15.097493,56.630001,38.799381,38.278877,39.478432,31.914127,79.791260,91.370003,41.963512
2014/2/21,34.994877,52.563938,209.600006,90.080269,66.352722,5.962536,37.080666,44.670025,17.650284,3.69,...,47.974815,15.140164,55.919998,38.755493,37.602715,39.452480,31.959721,79.506798,91.820000,42.505081
2014/2/24,34.084637,53.229931,217.649994,90.784317,66.643288,5.979546,37.053185,45.098660,17.622553,3.71,...,47.889507,15.097493,55.779999,39.018822,36.775414,39.867779,31.739365,80.686485,94.669998,43.088970


In [171]:
df.head()


,ATHM,V,TSLA,VIS,AAPL,ABEV,CF,AIG,AMAT,AMD,...,TGT,TSM,TWTR,TXN,VZ,WFC,WMB,XOM,YELP,VLO
Date,,,,,,,,,,,,,,,,,,,,,
2014/2/18,35.825970,53.199341,203.699997,89.915695,68.972733,5.715869,34.519791,45.755302,17.419250,3.70,...,48.102768,15.182836,58.180000,38.632580,36.576538,39.911037,30.804720,78.703629,91.739998,42.936657
2014/2/19,35.994167,52.686302,193.639999,89.138504,67.883812,5.732880,36.264172,44.916264,17.326839,3.72,...,48.887573,15.157234,55.500000,38.562363,37.014050,39.391911,30.774330,78.603218,90.000000,42.733562
2014/2/20,34.886044,52.587467,209.970001,90.107704,67.098045,5.928513,36.758640,44.888901,17.548626,3.69,...,48.298965,15.097493,56.630001,38.799381,38.278877,39.478432,31.914127,79.791260,91.370003,41.963512
2014/2/21,34.994877,52.563938,209.600006,90.080269,66.352722,5.962536,37.080666,44.670025,17.650284,3.69,...,47.974815,15.140164,55.919998,38.755493,37.602715,39.452480,31.959721,79.506798,91.820000,42.505081
2014/2/24,34.084637,53.229931,217.649994,90.784317,66.643288,5.979546,37.053185,45.098660,17.622553,3.71,...,47.889507,15.097493,55.779999,39.018822,36.775414,39.867779,31.739365,80.686485,94.669998,43.088970


In [172]:
cdx = df.iloc[:,0]
cdx.head()

Date
2014/2/18    35.825970
2014/2/19    35.994167
2014/2/20    34.886044
2014/2/21    34.994877
2014/2/24    34.084637
Name: ATHM, dtype: float64

In [173]:
single_cds = df.iloc[:,1:]
single_cds.head()

,V,TSLA,VIS,AAPL,ABEV,CF,AIG,AMAT,AMD,AR,...,TGT,TSM,TWTR,TXN,VZ,WFC,WMB,XOM,YELP,VLO
Date,,,,,,,,,,,,,,,,,,,,,
2014/2/18,53.199341,203.699997,89.915695,68.972733,5.715869,34.519791,45.755302,17.419250,3.70,55.017555,...,48.102768,15.182836,58.180000,38.632580,36.576538,39.911037,30.804720,78.703629,91.739998,42.936657
2014/2/19,52.686302,193.639999,89.138504,67.883812,5.732880,36.264172,44.916264,17.326839,3.72,56.192242,...,48.887573,15.157234,55.500000,38.562363,37.014050,39.391911,30.774330,78.603218,90.000000,42.733562
2014/2/20,52.587467,209.970001,90.107704,67.098045,5.928513,36.758640,44.888901,17.548626,3.69,55.498985,...,48.298965,15.097493,56.630001,38.799381,38.278877,39.478432,31.914127,79.791260,91.370003,41.963512
2014/2/21,52.563938,209.600006,90.080269,66.352722,5.962536,37.080666,44.670025,17.650284,3.69,55.017555,...,47.974815,15.140164,55.919998,38.755493,37.602715,39.452480,31.959721,79.506798,91.820000,42.505081
2014/2/24,53.229931,217.649994,90.784317,66.643288,5.979546,37.053185,45.098660,17.622553,3.71,56.105583,...,47.889507,15.097493,55.779999,39.018822,36.775414,39.867779,31.739365,80.686485,94.669998,43.088970


In [174]:
def log_return(price_df):
    '''
    This function calculates the logrithm return
    
    Input:
    - df : adj closed price of assets

    Output:
    - log-return of assets
    '''
    return np.log(price_df) - np.log(price_df.shift())


log_ret = log_return(single_cds)
log_ret.head()

,V,TSLA,VIS,AAPL,ABEV,CF,AIG,AMAT,AMD,AR,...,TGT,TSM,TWTR,TXN,VZ,WFC,WMB,XOM,YELP,VLO
Date,,,,,,,,,,,,,,,,,,,,,
2014/2/18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014/2/19,-0.009691,-0.050648,-0.008681,-0.015914,0.002972,0.049297,-0.018508,-0.005319,0.005391,0.021126,...,0.016184,-0.001688,-0.047159,-0.001819,0.011891,-0.013092,-0.000987,-0.001277,-0.019149,-0.004741
2014/2/20,-0.001878,0.080964,0.010814,-0.011643,0.033555,0.013543,-0.000609,0.012719,-0.008097,-0.012414,...,-0.012113,-0.003949,0.020156,0.006128,0.033601,0.002194,0.036368,0.015001,0.015108,-0.018184
2014/2/21,-0.000448,-0.001764,-0.000305,-0.011170,0.005722,0.008722,-0.004888,0.005776,0.000000,-0.008712,...,-0.006734,0.002822,-0.012617,-0.001132,-0.017822,-0.000658,0.001428,-0.003571,0.004913,0.012823
2014/2/24,0.012591,0.037687,0.007785,0.004370,0.002849,-0.000741,0.009550,-0.001572,0.005405,0.019583,...,-0.001780,-0.002822,-0.002507,0.006772,-0.022247,0.010472,-0.006919,0.014729,0.030567,0.013643


In [175]:
cov_mat = log_ret.cov()
cov_mat.to_csv("./covariance_matrix.csv")


In [176]:
rr = 0.4
lam_default_intensity = single_cds / (1 - rr) / 10000
lam_default_intensity.head()


,V,TSLA,VIS,AAPL,ABEV,CF,AIG,AMAT,AMD,AR,...,TGT,TSM,TWTR,TXN,VZ,WFC,WMB,XOM,YELP,VLO
Date,,,,,,,,,,,,,,,,,,,,,
2014/2/18,0.008867,0.033950,0.014986,0.011495,0.000953,0.005753,0.007626,0.002903,0.000617,0.009170,...,0.008017,0.002530,0.009697,0.006439,0.006096,0.006652,0.005134,0.013117,0.015290,0.007156
2014/2/19,0.008781,0.032273,0.014856,0.011314,0.000955,0.006044,0.007486,0.002888,0.000620,0.009365,...,0.008148,0.002526,0.009250,0.006427,0.006169,0.006565,0.005129,0.013101,0.015000,0.007122
2014/2/20,0.008765,0.034995,0.015018,0.011183,0.000988,0.006126,0.007481,0.002925,0.000615,0.009250,...,0.008050,0.002516,0.009438,0.006467,0.006380,0.006580,0.005319,0.013299,0.015228,0.006994
2014/2/21,0.008761,0.034933,0.015013,0.011059,0.000994,0.006180,0.007445,0.002942,0.000615,0.009170,...,0.007996,0.002523,0.009320,0.006459,0.006267,0.006575,0.005327,0.013251,0.015303,0.007084
2014/2/24,0.008872,0.036275,0.015131,0.011107,0.000997,0.006176,0.007516,0.002937,0.000618,0.009351,...,0.007982,0.002516,0.009297,0.006503,0.006129,0.006645,0.005290,0.013448,0.015778,0.007181


In [177]:
lam_default_intensity_corr = lam_default_intensity.corr()


In [178]:
lam_default_intensity_adj =  lam_default_intensity @ lam_default_intensity_corr
default_cov_mat = lam_default_intensity_adj.cov()

In [179]:
w,v = np.linalg.eig(cov_mat)  # w: vector of eigenvalues, v:eigenvector
# w: vector of eigenvalues, v:eigenvector
idx = w.argsort()[::-1]
w = w[idx]
v = v[:,idx]



In [180]:
# count how many positive eigenvalues in the matrix decomposition
num = [1 for i in range(len(w)) if w[i]>0]

In [181]:
explianed_var = [np.abs(i)/np.sum(w) for i in w]
sum_var = np.cumsum(explianed_var)
def find(sort_var,l):  #a is required variance
    i=0
    while sort_var[i]<l:
        i+=1
    return i

n1 = find(sum_var,0.5)+1     # 9 of eigenvalues explianed 50% of the variance
n2 = find(sum_var,0.9)+1     # 70 of eigenvalues explianed 90% of the variance

In [182]:
tickers = single_cds.columns
########### compute the vectors from svd   ##########################
svd1, svd_diag, svd2 = np.linalg.svd(cov_mat)

#print(svd1)
svd_diag_stable = list(svd_diag)[:n2]
svd_diag_recirocal = [1/i for i in svd_diag_stable]
# print(len(svd_diag_recirocal))
rest = np.zeros(len(tickers) - len(svd_diag_recirocal))
svd_all = np.append(svd_diag_recirocal, rest)
# print(svd_all)
diag_stable_cov = np.diag(svd_all)
# print(diag_stable_cov)


In [183]:
# Parameters:
# Matrix:
# G: constriant matrix, size: N*2
# C: covariance matrix of returns, size: N*N
# R: expected return matrix of N companies
# c: the RHS of Constriants, size: 2*1
# a: parameter in a<w,Cw>, scalar

##################################################################

N = len(tickers)  # number of single name CDS

#  Form G matrix
g1 = np.ones(N)        # g1: budget constriant
g2_1 = np.ones(17)     # g2: constriant on the first 17 parameters
g2_2 = np.zeros(N-17)
# G = np.matrix([g1])
G = np.matrix([g1, np.append(g2_1, g2_2)])
# print(G)


# compute the inverse of G(C.I)G.T
C_inverse = svd1 * diag_stable_cov * svd2
inverse = np.linalg.inv(G * C_inverse * G.T)
# print()
# print(inverse)


a = 1  # the parameter in a<w,Cw>

# R is the expected return matrix of the N companies, R size: N*1
R = (np.matrix(np.mean(log_ret))).T
# c = np.matrix([1])
c = np.matrix([[1], [0.1]])    # RHS of constriants

# compute the value of lambda
part1 = G * C_inverse * R - 2 * a * c
# print(part1)
# print()
lam = inverse * part1
# print(lam)


# compute weights of portfolio
weights = (1/(2*a)) * C_inverse * (R - G.T * lam)
# print(weights)

up = max(weights)
low = min(weights)
print(up)   # maximum weight
print(low)  # minimum weight


[[0.48984993]]
[[-0.05144857]]


/usr/local/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3417: FutureWarning: In a future version, DataFrame.mean(axis=None) will return a scalar mean over the entire DataFrame. To retain the old behavior, use 'frame.mean(axis=0)' or just 'frame.mean()'
  return mean(axis=axis, dtype=dtype, out=out, **kwargs)


In [184]:
pd.DataFrame(weights).to_csv('./weights.csv')

In [185]:
np.exp(np.cumsum(log_ret @ weights))

,0
Date,
2014/2/18,NaN
2014/2/19,0.987164
2014/2/20,0.988571
2014/2/21,0.980119
2014/2/24,0.993026
...,...
2019/2/8,1.438238
2019/2/11,1.448317
2019/2/12,1.434420
